# Style Transfer

An example below is inspired [by this blog post](https://medium.com/mlreview/making-ai-art-with-style-transfer-using-keras-8bb5fa44b216), and a lot of code is borrowed from there. Another good example of Style Transfer using CNTK framework is [here](https://github.com/Microsoft/CNTK/blob/master/Tutorials/CNTK_205_Artistic_Style_Transfer.ipynb). Here is an original paper on [Artistic Style Transfer](https://arxiv.org/abs/1508.06576).

Main ideas behind style transfer are the following:

* Starting from white noise, we try to optimize the current image $x$ to minimize some loss function
* Loss function consists of three components $\mathcal{L(x)} = \alpha\mathcal{L}_c(x,i) + \beta\mathcal{L}_s(x,s)+\gamma\mathcal{L}_t(x)$
   - $\mathcal{L}_c$ - content loss - shows how close the current image $x$ is to original image $i$
   - $\mathcal{L}_s$ - style loss - shows how close the current image $x$ is to style image $s$
   - $\mathcal{L}_t$ - total variation loss (we will not consider it in our example) - makes sure that the resulting image is smooth, i.e. it shows the mean squared error of neighbouring pixels of the image $x$
   
Those loss functions have to be designed in a clever way, so that for example style loss corresponds to styles of the images being similar, and not the actual content. For that, we will compare some deeper feature layers of a CNN which looks at the image.

Let's start by loading a couple of images:

In [ ]:
import os
import tensorflow as tf

# Example images and attribution: https://www.tensorflow.org/tutorials/generative/style_transfer
# Content: Yellow Labrador Looking, Elf / Wikimedia Commons, CC BY-SA 3.0.
content_path = tf.keras.utils.get_file('YellowLabradorLooking_new.jpg',
    'https://storage.googleapis.com/download.tensorflow.org/example_images/YellowLabradorLooking_new.jpg')
style_path = tf.keras.utils.get_file('kandinsky_composition7.jpg',
    'https://storage.googleapis.com/download.tensorflow.org/example_images/Vassily_Kandinsky%2C_1913_-_Composition_7.jpg')


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import fmin_l_bfgs_b


Load and resize the images to 256×256. This notebook uses TensorFlow eager execution and GradientTape. The optional STYLE_TRANSFER_SIZE and STYLE_TRANSFER_ITERATIONS environment variables allow a smaller compatibility smoke test; those results are not full optimization or visual-quality validation.


In [ ]:
img_size = int(os.environ.get('STYLE_TRANSFER_SIZE', '256'))
content_steps = int(os.environ.get('STYLE_TRANSFER_ITERATIONS', '30'))
style_steps = int(os.environ.get('STYLE_TRANSFER_ITERATIONS', '10'))

def load_image(filename):
    return tf.keras.utils.img_to_array(tf.keras.utils.load_img(
        filename, target_size=(img_size, img_size)))

img_style = load_image(style_path)
img_content = load_image(content_path)
img_result = np.random.default_rng(42).uniform(0, 255, (img_size, img_size, 3)).astype(np.float32)
fig, ax = plt.subplots(1, 3)
for axis, image in zip(ax, [img_content, img_style, img_result]):
    axis.imshow(image.astype(np.uint8))
plt.show()


In [ ]:
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.applications import VGG16


In [ ]:
img_content_var = tf.constant(preprocess_input(img_content[None].copy()), dtype=tf.float32)
img_style_var = tf.constant(preprocess_input(img_style[None].copy()), dtype=tf.float32)
img_result_1 = preprocess_input(img_result[None].copy())


To calculate style loss and content loss, we need to work in the feature space extracted by a CNN. We can use different CNN architectures, but for simplicity in our case we will chose VGG-16, pre-trained on *ImageNet*.

In [ ]:
gModel = VGG16(include_top=False, weights='imagenet', input_shape=(img_size, img_size, 3))
gModel.trainable = False
cModel = sModel = gModel
feature_extractors = {}


Let's have a look at the model architecture:

In [ ]:
gModel.summary()

## Content Loss

**Content loss** will show how close our current image $x$ is to the original image. It looks at the intermediate feature layers in CNN, and computes square error. Content loss on layer $l$ will be defined as
$$
\mathcal{L}_c = {1\over2}\sum_{i,j} (F_{ij}^{(l)}-P_{ij}^{(l)})^2
$$
where $F^{(l)}$ and $P^{(l)}$ -- features at layer $l$.

In [ ]:
def get_feature_reps(x, layer_names, model):
    key = (id(model), tuple(layer_names))
    if key not in feature_extractors:
        feature_extractors[key] = tf.keras.Model(model.input,
            [model.get_layer(name).output for name in layer_names])
    features = feature_extractors[key](tf.convert_to_tensor(x, dtype=tf.float32), training=False)
    if not isinstance(features, (list, tuple)):
        features = [features]
    return [tf.transpose(tf.reshape(feature, (-1, int(feature.shape[-1]))))
            for feature in features]

def get_content_loss(F, P):
    return 0.5 * tf.reduce_sum(tf.square(F - P))

def loss_and_gradient(loss_function, image):
    image = tf.convert_to_tensor(image.reshape((1, img_size, img_size, 3)), dtype=tf.float32)
    with tf.GradientTape() as tape:
        tape.watch(image)
        loss = loss_function(image)
    gradient = tape.gradient(loss, image)
    if gradient is None:
        raise RuntimeError('Image gradient is disconnected')
    return float(loss.numpy()), gradient.numpy().astype(np.float64).ravel()


To inspect a layer's content representation, optimize the generated image with SciPy L-BFGS. TensorFlow GradientTape differentiates the loss with respect to the image while keeping the pretrained network fixed.


In [ ]:
layer = 'block4_conv2'
P = get_feature_reps(img_content_var, [layer], cModel)[0]
def helper_loss(image):
    return get_content_loss(get_feature_reps(image, [layer], gModel)[0], P)
def help_loss(image):
    return loss_and_gradient(helper_loss, image)
x, _, _ = fmin_l_bfgs_b(help_loss, img_result_1.ravel().astype(np.float64), maxiter=content_steps)


In [ ]:
def postprocess_array(x):
    # Zero-center by mean pixel
    if x.shape != (img_size, img_size, 3):
        x = x.reshape((img_size, img_size, 3))
    x[..., 0] += 103.939
    x[..., 1] += 116.779
    x[..., 2] += 123.68
    # 'BGR'->'RGB'
    x = x[..., ::-1]
    x = np.clip(x, 0, 255)
    x = x.astype('uint8')
    return x


In [ ]:
plt.imshow(postprocess_array(x.copy()))
plt.show()

In [ ]:
layer='block3_conv2'
P = get_feature_reps(x=img_content_var, layer_names=[layer], model=cModel)[0]
x = img_result_1.ravel().astype(np.float64)
x, _, _ = fmin_l_bfgs_b(help_loss, x, maxiter=content_steps)
plt.imshow(postprocess_array(x.copy()))
plt.show()

In [ ]:
layer='block5_conv1'
P = get_feature_reps(x=img_content_var, layer_names=[layer], model=cModel)[0]
x = img_result_1.ravel().astype(np.float64)
x, _, _ = fmin_l_bfgs_b(help_loss, x, maxiter=content_steps)
plt.imshow(postprocess_array(x.copy()))
plt.show()

## Style Loss


Style loss is the main idea behind Style Transfer. We compare not the actual features, but their Gram matrices, which are defined as $$G=A\times A^T$$

Gram matrix is similar to correlation matrix, and it shows how some filters depend on the others. Style Loss is computed as a sum of losses from different layers, which are often considered with weighted coefficients.

Total loss function for style transfer is a sum of *content loss* and *style loss*.

In [ ]:
def get_Gram_matrix(F):
    G = tf.matmul(F, tf.transpose(F))
    return G

def get_style_loss(ws, Gs, As):
    sLoss = tf.constant(0., dtype=tf.float32)
    for w, G, A in zip(ws, Gs, As):
        M_l = G.shape[1]
        N_l = G.shape[0]
        G_gram = get_Gram_matrix(G)
        A_gram = get_Gram_matrix(A)
        sLoss+= w*0.25*tf.reduce_sum(tf.square(G_gram - A_gram))/ (N_l**2 * M_l**2)
    return sLoss
  
def get_total_loss(gImPlaceholder, alpha=1.0, beta=30.0):
    F = get_feature_reps(gImPlaceholder, layer_names=[content_layer_name], model=gModel)[0]
    Gs = get_feature_reps(gImPlaceholder, layer_names=style_layer_names, model=gModel)
    contentLoss = get_content_loss(F, P)
    styleLoss = get_style_loss(ws, Gs, As)
    totalLoss = alpha*contentLoss + beta*styleLoss
    return totalLoss
 

## Putting it all together

Here `calualate_loss` function will calculate total loss:

In [ ]:
def calculate_loss(image):
    return loss_and_gradient(get_total_loss, image)

content_layer_name = 'block4_conv2'
style_layer_names = ['block1_conv1', 'block2_conv1', 'block3_conv1', 'block4_conv1']
P = get_feature_reps(img_content_var, [content_layer_name], cModel)[0]
As = get_feature_reps(img_style_var, style_layer_names, sModel)
ws = np.ones(len(style_layer_names)) / len(style_layer_names)
iterations = style_steps
x_opt = img_result_1.ravel().astype(np.float64)


The code below performs the actual optimization of loss. Keep in mind that even with GPU the optimization takes significant amount of time. You can run the cell below several times to improve the result.

In [ ]:
xopt, f_val, info= fmin_l_bfgs_b(calculate_loss, x_opt, maxiter=iterations, disp=True)
plt.imshow(postprocess_array(xopt.copy()))
plt.show()

In [ ]:
iterations = int(os.environ.get('STYLE_TRANSFER_ITERATIONS', '20'))
xopt, f_val, info= fmin_l_bfgs_b(calculate_loss, xopt,
                            maxiter=iterations, disp=True)
plt.imshow(postprocess_array(xopt.copy()))
plt.show()

## Add variation loss

**Variation loss** allows us to make the image less noisy, by minimizing the amount of difference between neighbouring pixels.

In [ ]:
def total_variation_loss(x):
    a = tf.square(x[:,:img_size-1,:img_size-1,:] - x[:, 1:, :img_size-1,:])
    b = tf.square(x[:,:img_size-1,:img_size-1,:] - x[:, :img_size-1, 1:,:])
    return tf.reduce_sum(tf.pow(a + b, 1.25))
  
def get_total_loss(gImPlaceholder, alpha=1.0, beta=30.0):
    F = get_feature_reps(gImPlaceholder, layer_names=[content_layer_name], model=gModel)[0]
    Gs = get_feature_reps(gImPlaceholder, layer_names=style_layer_names, model=gModel)
    contentLoss = get_content_loss(F, P)
    styleLoss = get_style_loss(ws, Gs, As)
    variationLoss = total_variation_loss(gImPlaceholder)
    totalLoss = alpha*contentLoss + beta*styleLoss + variationLoss
    return totalLoss

img_result = np.random.randint(256,size=(img_size,img_size,3)).astype(np.float64)
img_result_1 = preprocess_input(np.expand_dims(img_result, axis=0))
iterations = style_steps
x_opt = img_result_1.flatten()


In [ ]:
xopt, f_val, info= fmin_l_bfgs_b(calculate_loss, x_opt, maxiter=iterations, disp=True)
plt.imshow(postprocess_array(xopt.copy()))
plt.show()